# Multilingual Text Embedding on Intel® NPU with OpenVINO™

This notebook shows how to run a multilingual sentence-embedding model on the
Intel® NPU (Core Ultra series) using OpenVINO™. The NPU is well suited for
always-on embedding workloads: it runs at roughly 1 W, leaving the CPU and GPU
free for other tasks such as local LLM inference.

The key practical detail is that the NPU compiler **rejects dynamic input
shapes**. We solve this by reshaping the OpenVINO IR to a fixed shape
`[1, 512]` before compilation.

We use [`intfloat/multilingual-e5-small`](https://huggingface.co/intfloat/multilingual-e5-small)
(117 M parameters, 100+ languages, dim 384), which is small enough for the NPU
and strong enough for retrieval tasks.

#### Table of contents:

- [Installation Instructions](#Installation-Instructions)
- [Install dependencies](#Install-dependencies)
- [Export the model to OpenVINO IR](#Export-the-model-to-OpenVINO-IR)
- [Reshape to a static shape for the NPU](#Reshape-to-a-static-shape-for-the-NPU)
- [Embedding helpers](#Embedding-helpers)
- [Latency benchmark](#Latency-benchmark)
- [Multilingual semantic search demo](#Multilingual-semantic-search-demo)
- [Where this fits](#Where-this-fits)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter
server to start. For details, please refer to
[Installation Guide](../../README.md).


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/npu-multilingual-embedding/npu-multilingual-embedding.ipynb" />

## Install dependencies

In [1]:
%pip install -q "openvino>=2025.0.0" "optimum[openvino]>=1.20.0" "transformers>=4.44.0" "faiss-cpu>=1.8.0" numpy

Note: you may need to restart the kernel to use updated packages.


## Export the model to OpenVINO IR

We export with `compile=False` because the NPU needs a static reshape before
compilation.

In [2]:
from pathlib import Path

from optimum.intel import OVModelForFeatureExtraction
from transformers import AutoTokenizer

MODEL_ID = "intfloat/multilingual-e5-small"
OV_DIR = Path("model")

if not OV_DIR.exists():
    print(f"Exporting {MODEL_ID} to OpenVINO IR (one-time, ~30 s) ...")
    model = OVModelForFeatureExtraction.from_pretrained(
        MODEL_ID, export=True, compile=False, device="CPU"
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model.save_pretrained(OV_DIR)
    tokenizer.save_pretrained(OV_DIR)
else:
    print(f"Reusing exported IR at {OV_DIR}")

Exporting intfloat/multilingual-e5-small to OpenVINO IR (one-time, ~30 s) ...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


<path> TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


## Reshape to a static shape for the NPU

The NPU compiler rejects dynamic dimensions, so we lock the input to
`[batch=1, seq_len=512]`. Shorter inputs are still processed correctly via
padding and the attention mask.

If the NPU is unavailable, the same flow runs on CPU as a fallback. This
keeps the notebook reproducible on any machine with OpenVINO.

In [3]:
import openvino as ov

STATIC_SEQ_LEN = 512

core = ov.Core()
available = core.available_devices
print("Available devices:", available)

device = "NPU" if "NPU" in available else "CPU"
print(f"Using device: {device}")

model = core.read_model(OV_DIR / "openvino_model.xml")

# Reshape every input to [1, STATIC_SEQ_LEN] so the NPU compiler accepts it.
static_shape = {}
for inp in model.inputs:
    name = next(iter(inp.get_names()))
    static_shape[name] = [1, STATIC_SEQ_LEN]
model.reshape(static_shape)

ov_config = {"PERFORMANCE_HINT": "LATENCY"}
if device == "NPU":
    ov_config["NPU_TURBO"] = "YES"

print(f"Compiling for {device} (first compile may take 30-90 s)...")
compiled = core.compile_model(model, device, ov_config)
print("Compiled.")

Available devices: ['CPU', 'GPU.0', 'GPU.1', 'NPU']
Using device: NPU
Compiling for NPU (first compile may take 30-90 s)...


Compiled.


## Embedding helpers

`multilingual-e5` expects a `passage:` prefix for documents and a `query:`
prefix for queries. We mean-pool over real (non-padding) tokens and L2-
normalize the result so that dot-product equals cosine similarity.

In [4]:
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(OV_DIR)


def _mean_pool(last_hidden: np.ndarray, attn_mask: np.ndarray) -> np.ndarray:
    mask = attn_mask[..., None].astype(np.float32)
    summed = (last_hidden * mask).sum(axis=1)
    count = mask.sum(axis=1).clip(min=1e-9)
    return summed / count


def _embed(text: str, prefix: str) -> np.ndarray:
    batch = tokenizer(
        prefix + text,
        padding="max_length",
        truncation=True,
        max_length=STATIC_SEQ_LEN,
        return_tensors="np",
    )
    feed = {
        "input_ids": batch["input_ids"].astype(np.int64),
        "attention_mask": batch["attention_mask"].astype(np.int64),
    }
    if "token_type_ids" in batch:
        feed["token_type_ids"] = batch["token_type_ids"].astype(np.int64)
    out = compiled(feed)
    last_hidden = next(iter(out.values()))
    pooled = _mean_pool(last_hidden, batch["attention_mask"])
    pooled /= np.linalg.norm(pooled, axis=-1, keepdims=True).clip(min=1e-9)
    return pooled[0].astype(np.float32)


def encode_passage(text: str) -> np.ndarray:
    return _embed(text, "passage: ")


def encode_query(text: str) -> np.ndarray:
    return _embed(text, "query: ")

The tokenizer you are loading from 'model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


## Latency benchmark

We warm up once and then time 50 single-text encodes. On NPU 3720 we measure
around 60-80 ms per text; on CPU it is typically faster, but the NPU has the
advantage of running independently of the CPU/GPU.

In [5]:
import time

warmup = encode_passage("warmup")

N = 50
t0 = time.perf_counter()
for _ in range(N):
    _ = encode_passage("a sample sentence used for latency measurement")
elapsed = time.perf_counter() - t0

print(f"Device: {device}")
print(f"Mean latency over {N} runs: {elapsed / N * 1000:.1f} ms")
print(f"Throughput: {N / elapsed:.1f} embeds/s")

Device: NPU
Mean latency over 50 runs: 29.8 ms
Throughput: 33.6 embeds/s


## Multilingual semantic search demo

We build a tiny FAISS index over a handful of multilingual sentences and
query it in three different languages. With `multilingual-e5-small`, the
best match should be language-agnostic.

In [6]:
import faiss

passages = [
    "Edge AI runs locally on your laptop.",
    "엣지 AI는 노트북에서 직접 실행됩니다.",
    "La cuisine italienne utilise beaucoup d'huile d'olive.",
    "Football is the most popular sport globally.",
    "OpenVINO accelerates inference on Intel hardware.",
    "오픈비노는 인텔 하드웨어에서 추론을 가속합니다.",
]

vectors = np.stack([encode_passage(p) for p in passages])
index = faiss.IndexFlatIP(vectors.shape[1])  # cosine via pre-normalized vectors
index.add(vectors)

queries = [
    "How do I run AI on a laptop?",
    "인텔 NPU 가속",
    "sport",
]

for q in queries:
    qv = encode_query(q)[None, :]
    scores, ids = index.search(qv, k=3)
    print(f"\nQuery: {q}")
    for s, i in zip(scores[0], ids[0]):
        print(f"  [{s:+.3f}] {passages[i]}")


Query: How do I run AI on a laptop?
  [+0.880] Edge AI runs locally on your laptop.
  [+0.847] 엣지 AI는 노트북에서 직접 실행됩니다.
  [+0.819] OpenVINO accelerates inference on Intel hardware.

Query: 인텔 NPU 가속
  [+0.857] 오픈비노는 인텔 하드웨어에서 추론을 가속합니다.
  [+0.818] 엣지 AI는 노트북에서 직접 실행됩니다.
  [+0.807] OpenVINO accelerates inference on Intel hardware.

Query: sport
  [+0.844] Football is the most popular sport globally.
  [+0.778] OpenVINO accelerates inference on Intel hardware.
  [+0.766] Edge AI runs locally on your laptop.


## Where this fits

Once the embedding pipeline is on the NPU, it can run continuously alongside
a local LLM (for example, a model served by `llama-server` or `ovms`). A
common pattern is:

- The NPU encodes every user-assistant turn into a 384-dim vector and keeps
  it in a persistent FAISS index on disk.
- When a new prompt arrives, the NPU retrieves the top-k most similar past
  turns in roughly 50-100 ms.
- The retrieved snippets are injected into the prompt sent to the LLM.

Because the NPU has its own dedicated compute and runs at very low power,
this memory layer does not contend with the GPU/CPU that the LLM is using.

## License

This notebook is released under Apache 2.0. The underlying
`multilingual-e5-small` model is MIT-licensed.